# Head-to-head on the base paper's protocol (Kaggle GPU)

Runs naive RAG, the paper's three-agent rewrite, our adaptive layer, and the hybrid
(paper rewrite as our high tier) on the paper's own attack prompts, a stepwise attack,
and held-out patient questions. Every method uses the same local model.

**Setup**
1. On your PC run `python scripts/package_lab_submission.py`, then upload
   `dist/adaptive_privacy_rag_lab.zip` as a Kaggle Dataset (Datasets -> New Dataset).
2. Create a notebook, *File -> Import Notebook* with this file, then *Add Input* -> your dataset.
3. *Settings*: Accelerator **GPU T4 x2** (or P100), **Internet on** (needs a phone-verified account).
4. Run the cells in order. For the full run use *Save Version -> Save & Run All* so it keeps
   running in the background (12-hour limit) and the results zip is kept in the Output tab.

Answers and rewrites are cached in `/kaggle/working`, so re-running a cell in the same
session resumes instead of starting over.

In [ ]:
import glob, os, shutil, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
matches = glob.glob("/kaggle/input/**/run_pipeline.py", recursive=True)
assert matches, "Add the uploaded project zip as notebook input first."
PROJECT = "/kaggle/working/ppr"
if not os.path.exists(PROJECT):
    shutil.copytree(os.path.dirname(matches[0]), PROJECT)
os.chdir(PROJECT)
print(sorted(os.listdir()))

In [ ]:
%%bash
# Ollama serves an OpenAI-compatible API, so the project code runs unchanged.
apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import time, urllib.request
MODEL = "llama3.1:8b"  # the paper reports LLaMA-3-8B as one of its six backbones
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_NUM_PARALLEL="4", OLLAMA_KEEP_ALIVE="24h")
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
subprocess.run(["ollama", "pull", MODEL], check=True)

In [ ]:
!pip -q install openai pyyaml tqdm pandas pyarrow
import yaml
os.environ["OLLAMA_API_KEY"] = "ollama"  # Ollama ignores the key; the OpenAI client needs one
cfg = yaml.safe_load(open("configs/agents_config.yaml"))
cfg["llm_settings"]["mode"] = "api"
api = cfg["llm_settings"]["api"]
api.update(base_url="http://localhost:11434/v1", model=MODEL, api_key_env="OLLAMA_API_KEY",
           max_workers=4, max_retries=2)
api.pop("extra_body", None)  # reasoning_effort is for gpt-oss only
cfg["generation_params"]["max_new_tokens"] = 1500
yaml.safe_dump(cfg, open("configs/agents_config_kaggle.yaml", "w"))
print("Paper agents ->", api["base_url"], api["model"])

Quick check (a few minutes): tiny sizes, development half of the paper's prompts.

In [ ]:
!python scripts/evaluate_paper_protocol.py --model {MODEL} --agents-config configs/agents_config_kaggle.yaml     --corpus 12 --utility 4 --targeted 0:4 --untargeted 0:4 --stepwise 2     --output-dir results/adaptive_privacy/paper_protocol_smoke
print(open("results/adaptive_privacy/paper_protocol_smoke/report.md").read())

Full run: 300-record corpus, test half (prompts 125-249), 20 stepwise patients, 100 utility questions.

In [ ]:
!python scripts/evaluate_paper_protocol.py --model {MODEL} --agents-config configs/agents_config_kaggle.yaml     --corpus 300 --utility 100 --stepwise 20 --workers 4

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/adaptive_privacy/paper_protocol/report.md").read()))
shutil.make_archive("/kaggle/working/paper_protocol_results", "zip", "results/adaptive_privacy/paper_protocol")
print("Download /kaggle/working/paper_protocol_results.zip and unzip it into results/adaptive_privacy/paper_protocol/")